In [1]:
import os
import sys
import json
from datetime import datetime

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

# Set environment variables before importing transformers
os.environ["HUGGINGFACE_HUB_CACHE"] = (
    "/scratch/shareddata/dldata/huggingface-hub-cache/hub"  # Load local models
)
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HOME"] = (
    f"{os.environ["WRKDIR"]}/.cache/huggingface"  # Cache in work directory
)

import transformers
import numpy as np
from datasets import load_dataset

from utils.helpers import (
    get_system_prompt,
    get_default_response,
    make_prompt,
    parse_output,
    sample_dataset
)

# Force reload changes
from utils.prompts import *

from utils.constants import (
    PIPE_MAX_NEW_TOKENS,
    MODEL_TEMPERATURE,
    BATCH_SIZE,
    PIPE_RETURN_FULL_TEXT
)

from utils.plots import calculate_accuracy

In [2]:
LABELS = ["yes", "no"]  # Labels per column
EVAL_COLS = [
    "The exercise description matched the selected theme (Yes/No)",
    "The exercise description matched the selected topic (Yes/No)",
    "Included concepts that were too advanced (Yes/No)",
]
PRED_COLS = ["augmentedProblemDescription", "augmentedExampleSolution"]

DEFAULT_DATA = "./data/final_dataset.csv"
DEFAULT_MODEL = "Qwen/Qwen2.5-7B-Instruct"

In [3]:
task = "detect"
n_rows = 1
use_instructions = True
seed = 42
number_of_demonstrations = 0
type_of_demonstrations = 0

dataset = load_dataset("csv", data_files=DEFAULT_DATA, split="train", sep=";")
dataset = dataset.shuffle(seed=seed)

demonstrations = sample_dataset(
    dataset, seed, 0, type_of_demonstrations
)

system_prompt = get_system_prompt(task, demonstrations, bool(use_instructions))

dataset = dataset.map(
    lambda row: {
        "user_prompt": make_prompt(row, task),
        "system_prompt": system_prompt,
    },
)

if n_rows is not None and n_rows > 0:
    dataset = dataset.select(range(n_rows))

In [4]:
for row in dataset:
    print(row["system_prompt"])
    break

You are a system that evaluates programming exercises.

You will receive:
- a general theme of the exercise
- a more specific topic within the theme which the exercise should focus on
- a list of allowed programming concepts
- a programming exercise consisting of a problem description and an example solution written in Dart.

Your task:
Evaluate the exercise and decide whether the problem description adheres to the
provided theme and topic. You also need to decide whether the exercise utilizes
any programming concepts that are not present in the list of allowed concepts.

Counts as concepts:
- user input (e.g., stdin.readLineSync)
- program output (print)
- variables (declaring or storing values)
- arithmetics (+, -, *, /)
- conditional statements (if, else)
- logical operators (&&, ||)
- for loops
- while loops

Rules:
- A concept is "used" if it is present in the example solution.
- Basic syntax is ignored.
- Each concept must be explicitly matched to the allowed list.

CRITICAL OUTP

In [5]:
model = DEFAULT_MODEL
#model = 'mistralai/Mistral-Small-3.2-24B-Instruct-2506'

params = {
    "model": model,
    "device_map": 0,  # Force GPU
    "max_new_tokens": PIPE_MAX_NEW_TOKENS,
    "temperature": MODEL_TEMPERATURE,
}

pipeline = transformers.pipeline("text-generation", **params)
pipeline.tokenizer.pad_token = pipeline.tokenizer.eos_token
pipeline.model.config.pad_token_id = pipeline.model.config.eos_token_id

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [6]:
s = datetime.now()

default_response = get_default_response(task)
results = {key: [] for key in default_response.keys()}

batch_size = BATCH_SIZE
prompts = dataset["user_prompt"]
system_prompts = dataset["system_prompt"]

for i in range(0, len(prompts), batch_size):

    # Build batched conversation inputs
    batch_prompts = zip(system_prompts[i : i + batch_size], prompts[i : i + batch_size])
    batch_inputs = (
        [
            {"role": "system", "content": sp},
            {"role": "user", "content": up},
        ]
        for sp, up in batch_prompts
    )

    # Single batched forward pass
    outputs = pipeline(
        batch_inputs,
        return_full_text=PIPE_RETURN_FULL_TEXT,
        batch_size=BATCH_SIZE
    )

    # Process each output in the batch
    for output in outputs:
        text = output[0]["generated_text"]
        parsed = parse_output(text)

        for key, value in default_response.items():
            results[key].append(json.dumps(parsed.get(key, value)))

e = datetime.now()

print(f"Done. Elapsed time: {e - s}")

Both `max_new_tokens` (=250) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Done. Elapsed time: 0:00:02.646072


In [7]:
def print_row(row):
    print("##################")
    print("#  Ground-truth  #")
    print("##################")
    print("Theme correct: " + row[EVAL_COLS[0]])
    print("Topic correct: " + row[EVAL_COLS[1]])
    print("Uses advanced concepts: " + row[EVAL_COLS[2]])
    print("Original concept: " + row["concept"])
    print(row["system_prompt"])
    print(row["user_prompt"])

    print("##################")
    print("#  Prediction    #")
    print("##################")
    print("Theme correct: " + results["themeCorrect"][i])
    print("Topic correct: " + results["topicCorrect"][i])
    print("Uses advanced concepts: " + results["usesAdditionalConcepts"][i])
    print("Explanation: " + results["explanation"][i])
    print("---------------------------------------------------")
    print()


print_n_rows = 10
if True:
    for i, row in data.iloc[:print_n_rows, :].iterrows():
        print_row(row)

NameError: name 'data' is not defined

In [ ]:
print(results)